# Lesson 4.1 — Agent vs Workflow: Architecture and Decision Criteria

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Define the difference between a workflow and an agent
- Apply decision criteria to pick one for a task
- Describe manager/supervisor hierarchies and subagents
- Diagram a simple workflow and a simple agent loop

**You will produce:** A decision helper plus a minimal workflow and a minimal agent loop (offline-safe).

## Phase 4 Overview

Phase 4 covers **Domain 1 - Agents and Workflows (14.7%)** and **Domain 3 - Claude Code (3.1%)**: choosing agents vs workflows, building agents with the Agent SDK and custom loops, agent patterns and frameworks, and operating Claude Code.

By the end of this phase, you will have built:

- An agent-vs-workflow decision helper
- A custom agent loop / harness with memory
- A manager/subagent hierarchy
- Notes mapping Strands, LangGraph, and PydanticAI
- Claude Code config (CLAUDE.md, settings.json, slash commands)

### Prerequisites

- Completed Phases 1-3 (API, prompts/context, tools/MCP)
- `pip install anthropic`
- Optional: Claude Code CLI installed for Lesson 4.5

## Concepts

**Workflows** and **agents** are two ends of a spectrum:

- A **workflow** orchestrates Claude through **predefined**, code-controlled steps. The path is fixed; you know in advance which calls happen and in what order. Best when the task is well-understood and you want predictability.
- An **agent** lets Claude **decide** which steps/tools to take in a loop until the goal is met. Best for open-ended tasks where the path cannot be fully predetermined.

**Decision criteria**: predictability vs flexibility, need for dynamic tool selection, acceptable cost/latency, and auditability. **Hierarchies**: a **manager/supervisor** agent decomposes a goal and delegates to **subagents** (each with isolated context, per Phase 2), then synthesizes results - improving reliability on complex tasks.

> **Reference:** [Anthropic - Building effective agents](https://docs.anthropic.com/en/docs/agents-and-tools/building-effective-agents)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Decision helper

Encode the main criteria as a function.

In [ ]:
def agent_or_workflow(task):
    if task.get("path_known") and not task.get("dynamic_tools"):
        return "workflow (predictable, code-controlled steps)"
    if task.get("open_ended") or task.get("dynamic_tools"):
        return "agent (Claude decides steps in a loop)"
    return "workflow (default to the simpler, more predictable option)"

print(agent_or_workflow({"path_known": True}))
print(agent_or_workflow({"open_ended": True, "dynamic_tools": True}))

## Code-Along: Step 2 — A minimal workflow

Fixed steps: classify, then route - no model-chosen branching.

In [ ]:
def classify_then_route(text):
    label = "billing" if "refund" in text.lower() else "general"  # step 1 (could be a model call)
    route = {"billing": "->billing queue", "general": "->general queue"}[label]  # step 2
    return label, route

print(classify_then_route("I need a refund"))

## Code-Along: Step 3 — A minimal agent loop

The model decides whether to keep going (simulated offline).

In [ ]:
def agent_loop(goal, max_steps=3):
    steps = []
    for i in range(max_steps):
        # In a real agent, Claude decides the next action/tool each iteration.
        action = f"step {i+1} toward: {goal}"
        steps.append(action)
        if i == 1:  # pretend the model signals completion
            break
    return steps

print(agent_loop("draft and refine a release note"))

## Exercise

Write `recommend_architecture(task)` returning 'workflow', 'single agent', or 'manager + subagents' based on complexity flags (`parallelizable`, `subtasks`, `open_ended`). Add tests.

In [ ]:
def recommend_architecture(task):
    # TODO: many independent subtasks -> 'manager + subagents'
    # TODO: open_ended single goal -> 'single agent'
    # TODO: known fixed path -> 'workflow'
    pass

## Key Takeaways

- Workflows use predefined, code-controlled steps; agents let Claude choose steps in a loop.
- Prefer workflows for predictability; use agents for open-ended, dynamic tasks.
- Manager/supervisor agents delegate to subagents with isolated context.
- Default to the simplest architecture that meets the requirement.

---

**Next:** [Lesson 4.2 — Custom Agent Loops, Harnesses, and Memory](./lesson-4.2-custom-agent-loop.ipynb)